In [1]:
from pydoc import describe

import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)
import openpyxl
import datetime as dt
import re
import os
import copy





#132.4	133.8	133.4	133.0	134.2	134.6




In [2]:
run_query = True

In [3]:
def run_sql(filename, sub_list=[], connection=None, filename_is_query=False):
    """
    Run a SQL Query by reading from a .txt file, substituting values when required.
    Input:
    filename (str): File that we want to read. Usually a .txt file.
    sub_list (list of (str,str) tuples): Substitute each instance of the first element of the tuple for the second.
                                         Example: [('{max_mob}', '6')]
    """
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for sub in sub_list:
        text, var = sub
        query = query.replace(text, var)
#     print(query)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df

In [4]:
if run_query == True:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
                # with open("establish_temp_tables_query.txt", "r") as file:
                #     temp_table_queries = file.read()
                # # conn.execute(temp_table_queries.strip())
                # print('temptables query finished')
        df = run_sql('weekly_query', connection=conn)
        print('loss query finished')
        bl = run_sql('b2l_query', connection=conn)
        print('book to look query finished')
        tminusone = run_sql('tminusone_query', connection=conn)
        ste_df = run_sql('ste_query', connection=conn)
        print('STE query finished')
        ste_bl = run_sql('ste_b2l', connection=conn)
        print('STE b2l query finished')
        
        # Save to cache for faster reloads
        df.to_pickle('df_cache.pkl')
        bl.to_pickle('bl_cache.pkl')
        tminusone.to_pickle('tminusone_cache.pkl')
        ste_df.to_pickle('ste_df_cache.pkl')
        ste_bl.to_pickle('ste_bl_cache.pkl')
        print('Data cached to pickle files')
        

else:
    # Load from cache (instant!)
    df = pd.read_pickle('df_cache.pkl')
    bl = pd.read_pickle('bl_cache.pkl')
    tminusone = pd.read_pickle('tminusone_cache.pkl')
    ste_df = pd.read_pickle('ste_df_cache.pkl')
    ste_bl = pd.read_pickle('ste_bl_cache.pkl')
    print('Loaded from cache')
    


loss query finished
book to look query finished
STE query finished
STE b2l query finished
Data cached to pickle files


In [ ]:
df['application_received_dtm'] = pd.to_datetime(df['application_received_dtm'])

df['quarter'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('Q')
df['month'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('M')
df['week'] = pd.to_datetime(df['application_received_dtm']).dt.to_period('W-SAT') #end on saturday



bl['time'] = pd.to_datetime(bl['time'])

bl['quarter'] = pd.to_datetime(bl['time']).dt.to_period('Q')
bl['month'] = pd.to_datetime(bl['time']).dt.to_period('M')
bl['week'] = pd.to_datetime(bl['time']).dt.to_period('W-SAT') #end on saturday








# df['con_ltv_back2'] = df['con_ltv_back'] *10.0652

count_acc_num = df['account_number'].nunique()

# countsds

#calculating
df['bbltv'] = df['con_amount_financed_back'] / df['bb_value'].replace(0, np.nan)





# df['atf_vantage'] = df['con_amount_financed_back']


# df['discount'] = np.where(df['lob'] == 'ENT', df['ent_disc'], 0)


df['luxury_flag'] = np.where( df['con_amount_financed_back'] >= 75000 , 1, 0)


# df['income_cb'] = df['income_cb'].fillna(0)

df['total_income'] = df['income_cb'].fillna(0) + df['income_pb'].fillna(0)


#assuming annual inflation of 3.5%, the average since 2020
# df['inflation_income'] = df['total_income'] / (1.035**(2025 - df['application_received_dtm'].dt.year.fillna(2025)  ))

# Set reference date as the start of 2025
# ref_date =

# Calculate number of weeks between application date and reference date
weeks_diff = ((pd.Timestamp.today() - df['application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
#
#
df['inflation_income'] = df['total_income'] / ( (1.035 ** (1/52)) ** weeks_diff)
#








# df['vantage'] = pd.to_numeric(df['vantage'], errors='coerce')
df['vantage2'] = np.where( (df['vantage'] >= 300) & (df['vantage'] <= 850)  , df['vantage'], np.nan    )




#df['ent_disc']




#filtering


#folters out the actual df similar to WHERE statement1
df = df[ df['application_received_dtm'] >= '2019-10-01']
# df = df[ df['application_received_dtm'] <= '2026-01-02']

df = df[ df['con_amount_financed_back'] <= 75000]
df = df[ df['con_pti_back'] <= 0.6]
# df = df[ df['bbltv'] <= 10.0]
df = df[ (df['lob'] == 'MCY') | (df['lob'] == 'STE') | (df['bbltv'] <= 10.0) | (df['bb_value'].isna()) | (df['bb_value'] == 0) ]
df = df[ df['total_income'] <= 200000]



# df = df[ df['bb_value'] > 1]
# df = df[ df['total_income'] <= 20000]




# df = df[(df['application_received_dtm'] > '2025-01-01')]





#if condition is TRUE then nan, else keep original value
# df['bbltv2'] = df['bbltv'].mask( (df['bbltv'] < 0.1) | (df['bbltv'] > 0.2) , np.nan)


#
# df['discount'] = np.select( [df['lob'] == 'ENT', df['data_source_id'] == 101]
#                             , [   df['ent_disc'] , df['aca_fee'] + df['processing_fee'] ]
#                             , np.nan)

df['discount'] = np.where( df['lob'] == 'ENT', df['ent_disc'] , df['disb_acquisition_fee_amt'] )


# --- STE cleaning ---
ste_df['application_received_dtm'] = pd.to_datetime(ste_df['application_received_dtm'])
ste_df['quarter'] = ste_df['application_received_dtm'].dt.to_period('Q')
ste_df['month'] = ste_df['application_received_dtm'].dt.to_period('M')
ste_df['week'] = ste_df['application_received_dtm'].dt.to_period('W-SAT')

ste_df['vantage2'] = ste_df['vantage']

ste_weeks_diff = ((pd.Timestamp.today() - ste_df['application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
ste_df['inflation_income'] = ste_df['total_income'] / ((1.035 ** (1/52)) ** ste_weeks_diff)

ste_df = ste_df[ ste_df['application_received_dtm'] >= '2019-10-01']
ste_df = ste_df[ ste_df['con_amount_financed_back'] <= 75000]
ste_df = ste_df[ ste_df['con_pti_back'] <= 0.6]
ste_df = ste_df[ ste_df['total_income'] <= 200000]

df = pd.concat([df, ste_df], ignore_index=True)


#STE preintegration income is wrong in los deal current fact so we make it payment / PTI
ste_pre = (df['lob'] == 'STE') & (df['application_received_dtm'] < '2025-10-07')
df.loc[ste_pre, 'total_income'] = df.loc[ste_pre, 'con_payment_back_amt'] / df.loc[ste_pre, 'con_pti_back']
ste_pre_weeks = ((pd.Timestamp.today() - df.loc[ste_pre, 'application_received_dtm'].fillna(pd.Timestamp.today())).dt.days // 7)
df.loc[ste_pre, 'inflation_income'] = df.loc[ste_pre, 'total_income'] / ((1.035 ** (1/52)) ** ste_pre_weeks)

ste_bl['time'] = pd.to_datetime(ste_bl['time'])
ste_bl['quarter'] = ste_bl['time'].dt.to_period('Q')
ste_bl['month'] = ste_bl['time'].dt.to_period('M')
ste_bl['week'] = ste_bl['time'].dt.to_period('W-SAT')

bl = pd.concat([bl, ste_bl], ignore_index=True)


df['lob_genre'] = np.select( [df['lob'] == 'KMX',
                              df['lob'] == 'ENT',
                              df['lob'] == 'STE',
                              (df['lob'] == 'AN') | (df['lob'] == 'STG') | (df['lob'] == 'FLD') | (df['lob'] == 'FRN') ]
                             , ['KMX', 'ENT', 'STE',
                                'NonKMXENT']
                            , np.nan)



bl['lob_genre'] = np.select( [bl['lob'] == 'KMX',
                              bl['lob'] == 'ENT',
                              bl['lob'] == 'STE',
                              (bl['lob'] == 'AN') | (bl['lob'] == 'STG') | (bl['lob'] == 'FLD') | (bl['lob'] == 'FRN') ]
                             , ['KMX', 'ENT', 'STE',
                                'NonKMXENT']
                            , np.nan)



df = df.dropna(subset=['lob_genre'])



mask = df['application_received_dtm'] >= '2023-01-01'
print(df.loc[mask, 'bbltv'].describe())

print(df['total_income'])



# df


count    410322.000000
mean          1.599317
std           0.542741
min           0.032702
25%           1.262907
50%           1.445722
75%           1.773999
max           6.986606
Name: bbltv, dtype: float64
0          2975.83
1          3966.26
2          6373.86
3         25559.49
4          2340.85
5          4779.38
6          5083.01
7          2000.00
8          2477.55
9          3654.20
10         2595.00
11         3046.29
12         8131.86
13         3069.12
14         2312.50
15         4041.68
16         2891.41
17         3442.24
18         1880.67
19         6720.00
20         3417.54
21         2293.20
22         5389.94
23         4513.88
24         3120.00
25         2295.79
26         2427.00
27         6846.67
28         1768.00
29         2773.36
            ...   
802565     7331.33
802566     5941.82
802567     4728.66
802568     4201.62
802569     5580.12
802570     6214.00
802571     7015.67
802572     4181.52
802573     5070.90
802574     9422.50
802575   

In [6]:
#aggregating

granularity = ['quarter','month' , 'week'][0]
lob_granularity = ['lob_genre', 'lob'][0]



def generate_report(df, bl, granularity, lob_granularity):
    # Find two Saturdays ago
    today = pd.Timestamp.today().normalize()
    days_since_saturday = (today.weekday() - 5) % 7
    last_saturday = today - pd.Timedelta(days=days_since_saturday)
    two_saturdays_ago = last_saturday - pd.Timedelta(days=7)

    # Get periods
    week_ref = pd.Period(two_saturdays_ago, freq='W-SAT')
    month_ref = pd.Period(two_saturdays_ago, freq='M')

    if granularity == 'quarter':
        min_quarter = pd.Period('2020Q1')
        df = df[df['quarter'] >= min_quarter]
    if granularity == 'month':
        last_5_months = month_ref - 4
        df = df[(df['month'] >= last_5_months) & (df['month'] <= month_ref)]
    if granularity == 'week':
        if lob_granularity == 'lob':
            last_n_weeks = 12  # 13 weeks total (current + 12 previous)
        else:
            last_n_weeks = 5   # 6 weeks total (current + 5 previous)
        last_weeks = week_ref - last_n_weeks
        df  = df[(df['week'] >= last_weeks) & (df['week'] <= week_ref)]


        # last_6_weeks = week_ref - 5
        # df = df[(df['week'] >= last_6_weeks) & (df['week'] <= week_ref)]




    quarterly_df = df.groupby([lob_granularity, granularity]).apply(
        lambda g: pd.Series({



            # 'vantage' : (g['vantage2'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'vantage': (g.loc[g['vantage2'].notnull(), 'vantage2'] * g.loc[g['vantage2'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['vantage2'].notnull(), 'con_amount_financed_back'].sum(),


            # 'vantage3' : g['vantage2'].sum(skipna=True),
            # 'vantage4' : g['vantage2'].mean(),

            # 'vantage5': np.average(g['vantage2'], weights=g['con_amount_financed_back']),
                         # * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            # 'model score weighted': (g['con_risk_model_score'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),
            'model score weighted': (g.loc[g['con_risk_model_score'].notnull(), 'con_risk_model_score'] * g.loc[g['con_risk_model_score'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['con_risk_model_score'].notnull(), 'con_amount_financed_back'].sum(),

            'cash down avg': g['con_cash_down_amt'].mean(),
            'cash down wtd': (g['con_cash_down_amt'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'discount avg': g['discount'].mean(),
            'discount pct': (g['discount']).sum() / g['con_amount_financed_back'].sum(),

            'amount financed avg': g['con_amount_financed_back'].mean(),

            'apr avg': g['con_apr'].mean(),
            'apr wtd': (g['con_apr'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'total income avg': g['total_income'].mean(),
            'inflation_adjusted_income avg': g['inflation_income'].mean(),

            'payment avg': g['con_payment_back_amt'].mean(),

            'pti wtd' : (g['con_pti_back']* g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),

            'contracts num': g['account_number'].nunique(),
            'duplicates': g['account_number'].duplicated().sum(),

            # 'contracts 2': len(g['account_number'].unique()),
        # .count(),
        #     'contracts 3': g['account_number'].drop_duplicates().count(),
        'all contracts': g['account_number'].count(),


    # VEHICLE METRICS
            'Blackbook Value avg': (g['bb_value'].mean()), #* g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum() ,
            'bbltv avg': g['bbltv'].mean(),
            # 'bbltv weighted': (g['bbltv'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum(),
            # 'bbltv 2weighted': (g.loc[g['bbltv'] > 1, 'bbltv'] * g.loc[g['bbltv'] > 1, 'con_amount_financed_back']).sum() / g.loc[g['bbltv'] > 1, 'con_amount_financed_back'].sum(),
            'bbltv 2weighted': (g.loc[g['bbltv'].notnull(), 'bbltv'] * g.loc[g['bbltv'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['bbltv'].notnull(), 'con_amount_financed_back'].sum(),

            'mileage avg': g['purchase_odometer'].mean(),
            'vehicle age avg':g['veh_age'].mean(),


        })
    ).reset_index()




    b2l_df = bl.groupby([lob_granularity, granularity]).apply(
        lambda g: pd.Series({

            # 'b2l avg': g['b2l'].mean(),
            'b2l': g['cons'].sum() / g['apps'].sum() if g['apps'].sum() > 0 else np.nan

        })
    ).reset_index()


    merged_df = pd.merge(quarterly_df, b2l_df, on=[lob_granularity, granularity], how='left')



    # Melt to long format: one row per lob_genre, quarter, measure, value
    melted = merged_df.melt(id_vars=[lob_granularity, granularity], var_name='measure', value_name='value')

    # Pivot so quarters are columns, measures are rows, lob_genre repeats for each measure
    pivoted = melted.pivot_table(index=[lob_granularity, 'measure'], columns=granularity, values='value')

    # Optional: reset index for a clean DataFrame
    pivoted = pivoted.reset_index()

    # List of measures in your desired order
    measure_order = [
        'vantage',# 'vantage3', 'vantage4',
        'model score weighted', 'cash down avg',
        'cash down wtd', 'discount avg', 'discount pct', 'amount financed avg',
        'apr avg', 'apr wtd', 'total income avg',

        'inflation_adjusted_income avg',

        'payment avg', 'pti wtd', 'contracts num',
        # 'duplicates', 'all contracts'
         'b2l',

        'Blackbook Value avg',  #'bbltv avg',
        'bbltv 2weighted', 'mileage avg',
        'vehicle age avg'
    ]

    # After melting
    melted['measure'] = pd.Categorical(melted['measure'], categories=measure_order, ordered=True)

    # Now pivot as before
    pivoted = melted.pivot_table(index=[lob_granularity, 'measure'], columns=granularity, values='value')


    pivoted.reset_index()

    return pivoted
    # pivoted















In [7]:


# generate_report(df, bl, 'quarter', 'lob_genre')
quarterly_df = generate_report(df, bl, 'quarter', 'lob_genre')

monthly_df = generate_report(df, bl, 'month', 'lob_genre')

weekly_df = generate_report(df, bl, 'week', 'lob_genre')

lob_breakout_df = generate_report(df, bl, 'week', 'lob')

quarterly_lob_breakout = generate_report(df, bl, 'quarter', 'lob')



quarterly_lob_breakout



C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_13824\1411491348.py:91: RuntimeWarning: invalid value encountered in scalar divide
  'bbltv 2weighted': (g.loc[g['bbltv'].notnull(), 'bbltv'] * g.loc[g['bbltv'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['bbltv'].notnull(), 'con_amount_financed_back'].sum(),
C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_13824\1411491348.py:145: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  melted['measure'] = pd.Categorical(melted['measure'], categories=measure_order, ordered=True)
C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_13824\1411491348.py:91: RuntimeWarning: invalid value encountered in scalar divide
  'bbltv 2weighted': (g.loc[g['bbltv'].notnull(), 'bbltv'] * g.loc[g['bbltv'].notnull(), 'con_amount_financed_back']).sum() / g.loc[g['bbltv'].notnull(), 'con_amount_financed_back'].sum(),
C:\Users\ahmed.

quarter                                          2020Q1         2020Q2  \
lob        measure                                                       
           vantage                           562.213894     549.132073   
           model score weighted              132.121113     138.329187   
           cash down avg                    5537.500000    2666.666667   
           cash down wtd                    7239.492100    2808.253077   
           discount avg                     3383.854287    2387.154833   
           discount pct                        0.205501       0.239482   
           amount financed avg             16466.391250    9968.000000   
           apr avg                             0.267850       0.257667   
           apr wtd                             0.273052       0.255919   
           total income avg                 4784.821250    5703.836667   
           inflation_adjusted_income avg    3871.657506    4640.381963   
           payment avg                       531.017500     376.976667   
           pti wtd                             0.123083       0.113689   
           contracts num                       8.000000       3.000000   
           Blackbook Value avg             12028.125000    8641.666667   
           bbltv 2weighted                     1.492488       1.418242   
           mileage avg                    129688.375000  183153.333333   
           vehicle age avg                    10.343750      11.277778   
AN         vantage                           529.762544     539.130346   
           model score weighted              131.683617     134.016185   
           cash down avg                    2567.063217    2918.436356   
           cash down wtd                    2668.396031    2996.229837   
           discount avg                     1562.255902    1756.290193   
           discount pct                        0.109465       0.130580   
           amount financed avg             14271.692662   13449.958084   
           apr avg                             0.249059       0.252525   
           apr wtd                             0.243893       0.249027   
           total income avg                 3584.468076    3602.342093   
           inflation_adjusted_income avg    2900.946098    2941.058451   
           payment avg                       391.730350     373.384734   
...                                                 ...            ...   
STG        apr avg                             0.252634       0.256197   
           apr wtd                             0.250916       0.255311   
           total income avg                 3534.666590    3616.950867   
           inflation_adjusted_income avg    2860.340938    2952.473390   
           payment avg                       395.325532     378.011199   
           pti wtd                             0.145304       0.138385   
           contracts num                    4830.000000    4395.000000   
           b2l                                 0.088808       0.087391   
           Blackbook Value avg              8020.154836    7713.191834   
           bbltv 2weighted                     1.960813       1.952929   
           mileage avg                     91742.663698   93402.999545   
           vehicle age avg                     7.483364       7.587050   
unassigned vantage                           543.764552     512.375139   
           model score weighted              134.863014     135.601459   
           cash down avg                    3238.285560    2660.000000   
           cash down wtd                    3522.405231    2666.761679   
           discount avg                     1938.061604    2113.470540   
           discount pct                        0.203464       0.301210   
           amount financed avg              9525.332974    7016.600000   
           apr avg                             0.264962       0.240000   
           apr wtd                             0.263797       0.228893   
    

In [8]:
# pivoted
#

In [9]:


# import pandas as pd

def format_time_columns(df):
    new_cols = []
    for col in df.columns:
        if isinstance(col, pd.Period):
            if col.freqstr == 'M':
                new_cols.append(col.start_time.strftime('%b %y'))
            elif col.freqstr.startswith('W'):
                new_cols.append(col.start_time.strftime('%d-%b-%y'))
            elif col.freqstr.startswith('Q'):
                new_cols.append(f"{col.year} Q{col.quarter}")
            else:
                new_cols.append(col)
        else:
            new_cols.append(col)
    df = df.copy()
    df.columns = new_cols
    return df

# Usage:
quarterly_fmt = format_time_columns(quarterly_df)
monthly_fmt = format_time_columns(monthly_df)
weekly_fmt = format_time_columns(weekly_df)
lob_breakout_fmt = format_time_columns(lob_breakout_df)
quarterly_lob_breakout_fmt = format_time_columns(quarterly_lob_breakout)













wb = openpyxl.load_workbook('pivoted_results.xlsx')
for sheet_name in ['lobgenre_quarter', 'lobgenre_month', 'lobgenre_week', 'lob_week', 'alllob_quarter']:
    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        for merge in list(ws.merged_cells.ranges):
            ws.merged_cells.remove(merge)
wb.save('pivoted_results.xlsx')
wb.close()

with pd.ExcelWriter('pivoted_results.xlsx', engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
    quarterly_fmt.to_excel(writer, sheet_name='lobgenre_quarter', index=True)
    monthly_fmt.to_excel(writer, sheet_name='lobgenre_month', index=True)
    weekly_fmt.to_excel(writer, sheet_name='lobgenre_week', index=True)
    lob_breakout_fmt.to_excel(writer, sheet_name='lob_week', index=True)
    quarterly_lob_breakout_fmt.to_excel(writer, sheet_name='alllob_quarter', index=True)

